In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 200)

In [2]:
gr = pd.read_excel(r"C:\Resource\All Projects\supplier-risk-decision\supplier-risk-decision-framework\1_data\01_raw\gr_history.xlsx")
ncr_all = pd.read_excel(r"C:\Resource\All Projects\supplier-risk-decision\supplier-risk-decision-framework\1_data\01_raw\ncr_all.xlsx")
ncr_oc = pd.read_excel(r"C:\Resource\All Projects\supplier-risk-decision\supplier-risk-decision-framework\1_data\01_raw\ncr_open_closed.xlsx", sheet_name="Sheet4")

In [3]:
print("GR shape:", gr.shape)
print("NCR all shape:", ncr_all.shape)
print("NCR open/closed shape:", ncr_oc.shape)

GR shape: (60753, 24)
NCR all shape: (10619, 71)
NCR open/closed shape: (148089, 33)


In [4]:
# remove space between column name
gr.columns = gr.columns.str.strip()
print("GR columns:")
print(gr.columns.tolist())

GR columns:
['Material', 'Material description', 'Plant', 'Storage location', 'Movement type', 'Special Stock', 'Material Document', 'Order', 'Material Doc.Item', 'Purchase order', 'Item', 'Posting Date', 'Quantity', 'Unit of Entry', 'Order Price Unit', 'Order Unit', 'Amt.in loc.cur.', 'User Name', 'Document Date', 'Vendor', 'Batch', 'Reservation', 'Document Header Text', 'Time of Entry']


In [5]:
ncr_all.columns = ncr_all.columns.str.strip()
print("NCR logistics columns:")


# remove 'range[]'
ncr_all.columns = (
    ncr_all.columns
    .str.replace("Range[", "", regex=False)
    .str.replace("]", "", regex=False)
    .str.strip()
)

print(ncr_all.columns.tolist())

NCR logistics columns:
['Exception', 'Completion by date', 'Notification', 'Material', 'Notification Type', 'Notification Status', 'Priority Text', 'Description', 'Reference number', 'Address number', 'Priority', 'Addit. device data', 'Serial Number', 'Reference Quantity', 'Complaint Quantity', 'Unit of Measure', 'Supplier', 'Changed On', 'Changed by', 'Time of change', 'Work Center', 'Plant for Work Ctr', 'Work center', 'Priority Type', 'Malfunction End', 'Malfunction Start', 'Malfunctn End (Time)', 'Start of Malfunctn (Time)', 'Processing time', 'Reference Date', 'Reference Time', 'Purchasing Group', 'Customer Ref. Date', 'Customer Reference', 'Batch', 'City', 'District', 'QM Order', 'Country Key', 'Purchasing Document', 'Item purchasing doc.', 'Purch. organization', 'MPN material', 'Created on', 'Created By', 'Created At', 'Report Type', 'Production order', 'Plan no. operations', 'Short Text', 'Operation', 'Notification origin', 'Supplier Material Number', 'Customer Material Number'

In [6]:
# remove space between column name
ncr_oc.columns = ncr_oc.columns.str.strip()
print("ncr_oc columns:")
print(ncr_oc.columns.tolist())

ncr_oc columns:
['Reference Number', 'Notification Number', 'Material Number', 'Material Description', 'Notification Description', 'Standard Task Description', 'Custom Task Description', 'Task Responsible', 'Task Release Date', 'Task Release Time', 'Cause Responsibility', 'Defect Code', 'Defect Location', 'Disposition', 'Material Value', 'Notification Created By', 'Notification Created Date', 'Notification QTY', 'Notification status', 'Notification Type', 'Project', 'Supplier Name', 'Task Status', 'Notification Completed By', 'Notification Completion Date', 'Change Date', 'Task Completed By', 'Task Completion Date', 'Task Completion Time', 'Cause Responsibility code', 'Dashboard Status', 'Task Responsible Dept', 'Age (Days)']


In [7]:
# remove 'range[]'
ncr_oc.columns = (
    ncr_oc.columns
    .str.replace("Range[", "", regex=False)
    .str.replace("]", "", regex=False)
    .str.strip()
)
print(ncr_oc.columns.tolist())

['Reference Number', 'Notification Number', 'Material Number', 'Material Description', 'Notification Description', 'Standard Task Description', 'Custom Task Description', 'Task Responsible', 'Task Release Date', 'Task Release Time', 'Cause Responsibility', 'Defect Code', 'Defect Location', 'Disposition', 'Material Value', 'Notification Created By', 'Notification Created Date', 'Notification QTY', 'Notification status', 'Notification Type', 'Project', 'Supplier Name', 'Task Status', 'Notification Completed By', 'Notification Completion Date', 'Change Date', 'Task Completed By', 'Task Completion Date', 'Task Completion Time', 'Cause Responsibility code', 'Dashboard Status', 'Task Responsible Dept', 'Age (Days)']


In [8]:
ncr_oc.shape

(148089, 33)

In [9]:
ncr_oc["Notification Number"].nunique()

20781

In [10]:
ncr_oc["Notification Number"].duplicated().sum()

127308

In [11]:
ncr_oc.groupby("Notification Number")["Supplier Name"].nunique().value_counts()

ncr_oc.groupby("Notification Number")["Notification Created Date"].nunique().value_counts()

ncr_oc.groupby("Notification Number")["Disposition"].nunique().value_counts().head()

ncr_oc.groupby("Notification Number")["Cause Responsibility"].nunique().value_counts().head()

Cause Responsibility
1    18898
0     1758
2      123
3        2
Name: count, dtype: int64

In [12]:
ncr_oc.groupby("Notification Number")["Disposition"].nunique().value_counts()

Disposition
1    20750
0       31
Name: count, dtype: int64

In [13]:
ncr_oc["Disposition"].value_counts().head(20)

Disposition
NCR-Spl Rwrk On Site    34027
NCR-RTV                 22977
CRRC MA Scrap           20450
NCR-Rework              15602
Welding Repair          11096
NCR-Repair              10933
NCR-RTV Scrap On Sit     5772
Field Service RTV        5629
Welding Rework           5391
FS RTV in place          4397
NCR-Prelim. Review       2137
PreNCR-NCR Request       2082
FS Spl Rwrk On Site      2011
NCR-Concession Req.      1635
Rework-Outside Vend.     1459
NCR-MRB                  1074
NCR-SCH                  1046
OIL-Rework                182
Field Service Repair       56
Documentation Reject       44
Name: count, dtype: int64

In [14]:
# Clean text
ncr_oc["disp_clean"] = (
    ncr_oc["Disposition"]
    .astype(str)
    .str.lower()
    .str.strip()
)

In [15]:
# Define disposition
def classify_disposition(text):
    if pd.isna(text):
        return None

    t = str(text).lower().strip()

    if "scrap" in t:
        return "scrap"
    elif "rtv" in t:
        return "rtv"
    elif "mrb" in t or "review" in t:
        return "mrb_review"
    elif "rework" in t or "repair" in t or "rwrk" in t:
        return "rework"
    else:
        return "other"

In [16]:
ncr_oc["disp_group"] = ncr_oc["disp_clean"].apply(classify_disposition)

In [17]:
# Check the output
ncr_oc["disp_group"].value_counts(dropna=False)

disp_group
rework        80757
rtv           33003
scrap         26222
other          4854
mrb_review     3253
Name: count, dtype: int64

In [18]:
# check some special disposition
ncr_oc.loc[
    ncr_oc["disp_clean"].str.contains("scrap|rtv", na=False),
    ["Disposition", "disp_group"]
].drop_duplicates().sort_values("Disposition").head(30)

,Disposition,disp_group
137,CRRC MA Scrap,scrap
32,FS RTV in place,rtv
3,Field Service RTV,rtv
24,NCR-RTV,rtv
2283,NCR-RTV Scrap On Sit,scrap


In [19]:
severity_map = {
    "rework": 1,
    "mrb_review": 2,
    "rtv": 3,
    "scrap": 4,
    "other": 0
}

ncr_oc["severity"] = ncr_oc["disp_group"].map(severity_map)

In [20]:
ncr_oc["severity"].value_counts()

severity
1    80757
3    33003
4    26222
0     4854
2     3253
Name: count, dtype: int64

In [21]:
ncr_oc[ncr_oc["severity"] >= 3].shape

(59225, 36)

In [22]:
def last_non_null(series):
    s = series.dropna()
    return s.iloc[-1] if not s.empty else None

In [23]:
ncr_event = (
    ncr_oc
    .groupby("Notification Number")
    .agg({
        "Supplier Name": "first",
        "Notification Created Date": "min",
        "Notification Completion Date": "max",
        "Notification QTY": "max",
        "Material Value": "max",
        "severity": "max",
        "Cause Responsibility": last_non_null
    })
    .reset_index()
)

In [24]:
ncr_event.shape

(20781, 8)

In [25]:
ncr_event.head()

,Notification Number,Supplier Name,Notification Created Date,Notification Completion Date,Notification QTY,Material Value,severity,Cause Responsibility
0,200000001,WABTEC Passenger Transit Corpo,43515,43515.0,0,0.00,0,None
1,200000012,Vulcan Metals Corporation,43525,43525.0,4,245781.36,0,None
2,200000013,Vulcan Metals Corporation,43525,43525.0,4,245781.36,0,None
3,200000017,ACE Employment Unlimited Corp,43529,43559.0,0,0.00,1,None
4,200000032,TransTech of South Carolina In,43559,43571.0,1,66.01,4,None


In [26]:
ncr_event.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20781 entries, 0 to 20780
Data columns (total 8 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Notification Number           20781 non-null  int64  
 1   Supplier Name                 20778 non-null  object 
 2   Notification Created Date     20781 non-null  int64  
 3   Notification Completion Date  19962 non-null  float64
 4   Notification QTY              20781 non-null  int64  
 5   Material Value                20781 non-null  float64
 6   severity                      20781 non-null  int64  
 7   Cause Responsibility          19023 non-null  object 
dtypes: float64(2), int64(4), object(2)
memory usage: 1.3+ MB


In [27]:
ncr_event["Notification Created Date"] = pd.to_datetime(
    ncr_event["Notification Created Date"],
    origin="1899-12-30",
    unit="D",
    errors="coerce"
)

ncr_event["Notification Completion Date"] = pd.to_datetime(
    ncr_event["Notification Completion Date"],
    origin="1899-12-30",
    unit="D",
    errors="coerce"
)

In [28]:
ncr_event[["Notification Created Date", "Notification Completion Date"]].head()

,Notification Created Date,Notification Completion Date
0,2019-02-19,2019-02-19
1,2019-03-01,2019-03-01
2,2019-03-01,2019-03-01
3,2019-03-05,2019-04-04
4,2019-04-04,2019-04-16


In [29]:
(
    ncr_event["Notification Completion Date"] < ncr_event["Notification Created Date"]
).sum()

0

In [30]:
ncr_event[["Notification Created Date", "Notification Completion Date"]].isna().mean()

Notification Created Date       0.000000
Notification Completion Date    0.039411
dtype: float64

## Severity Definition
- 0 = other / unclassified
- 1 = rework / repair
- 2 = mrb / review
- 3 = rtv
- 4 = scrap

In [31]:
ncr_event["severity"].value_counts(dropna=False).sort_index()

severity
0     1145
1    10227
2      756
3     4918
4     3735
Name: count, dtype: int64

In [32]:
# Support to define Threshold later
(ncr_event["severity"] >= 3).mean()

0.416389971608681

In [33]:
ncr_event["Cause Responsibility"].isna().mean()

0.08459650642413744

In [34]:
ncr_event["Cause Responsibility"].value_counts(dropna=False).head(20)

Cause Responsibility
Supplier                                  10322
Production                                 4861
None                                       1758
Preassembly                                1207
Supplier CRC (China)                        668
Logistics                                   643
Causal Supplier                             257
Customer                                    144
Procurement                                 140
Project Management                          123
Engineering                                 114
Testing                                      93
Technology                                   93
Field Service                                91
FMI / ECP - Supplier                         60
Design                                       41
FMI / ECP - CRRC MA                          29
Quality                                      27
Engineering/design                           19
Technology - Manufacturing Engineering       16
Name: count, dtype:

In [35]:
ncr_event["event_month"] = ncr_event["Notification Created Date"].dt.to_period("M")

In [36]:
ncr_event["cycle_days"] = (
    ncr_event["Notification Completion Date"] - ncr_event["Notification Created Date"]
).dt.days

In [37]:
ncr_event[["event_month", "cycle_days"]].head()

,event_month,cycle_days
0,2019-02,0.0
1,2019-03,0.0
2,2019-03,0.0
3,2019-03,30.0
4,2019-04,12.0


In [108]:
ncr_event.to_csv("../1_data/02_interim/ncr_event_table.csv", index=False)

In [ ]:
ncr_event["event_month"] = ncr_event["Notification Created Date"].dt.to_period("M")

In [ ]:
#Check
ncr_event[["Notification Created Date", "event_month"]].head()

,Notification Created Date,event_month
0,2019-02-19,2019-02
1,2019-03-01,2019-03
2,2019-03-01,2019-03
3,2019-03-05,2019-03
4,2019-04-04,2019-04


In [ ]:
# Count the amount of NCR and severity events group by Suppliers * Month
ncr_month = (
    ncr_event
    .groupby(["Supplier Name", "event_month"])
    .agg(
        ncr_count=("Notification Number", "count"),
        severe_ncr=("severity", lambda x: (x >= 3).sum())
    )
    .reset_index()
)

In [ ]:
#Check
ncr_month.head()
# ncr_month.shape

,Supplier Name,event_month,ncr_count,severe_ncr
0,3m Company,2023-01,1,1
1,ABB INC.,2023-10,1,0
2,ABB INC.,2025-08,1,0
3,ABB INC.,2025-10,2,0
4,ACE Employment Unlimited Corp,2019-03,1,0


In [ ]:
# Read GR dataset
gr.columns

Index(['Material', 'Material description', 'Plant', 'Storage location',
       'Movement type', 'Special Stock', 'Material Document', 'Order',
       'Material Doc.Item', 'Purchase order', 'Item', 'Posting Date',
       'Quantity', 'Unit of Entry', 'Order Price Unit', 'Order Unit',
       'Amt.in loc.cur.', 'User Name', 'Document Date', 'Vendor', 'Batch',
       'Reservation', 'Document Header Text', 'Time of Entry'],
      dtype='object')

In [ ]:
# Generate Month column in GR dataset
gr["Posting Date"] = pd.to_datetime(gr["Posting Date"])
gr["month"] = gr["Posting Date"].dt.to_period("M")

In [ ]:
# aggregate the exposure group by Vendor * Month
gr_month = (
    gr
    .groupby(["Vendor", "month"])
    .agg(
        gr_qty=("Quantity", "sum"),
        gr_count=("Material Document", "count")
    )
    .reset_index()
)

In [ ]:
#Check
gr_month.head()


,Vendor,month,gr_qty,gr_count
0,1.000200e+09,2025-02,48450.00,2719
1,1.000200e+09,2025-03,61657.00,3384
2,1.000200e+09,2025-04,19649.88,1701
3,1.000200e+09,2025-05,24521.54,1290
4,1.000200e+09,2025-06,34621.88,1022


In [ ]:
#Check
gr_month.shape

(702, 4)

## Current Status
- Built NCR event-level table from task-level records
- Created disposition grouping and severity classification
- Aggregated NCR by supplier-month
- Found key mismatch between GR vendor number and NCR supplier name
- Next step: obtain supplier mapping table (vendor number ↔ supplier name) before merging exposure and event data

In [ ]:
# add vendor id to ncr event sheet
ncr_event = ncr_event.merge(
    ncr_oc[["Notification Number", "Supplier Name"]].drop_duplicates(),
    on="Notification Number",
    how="left"
)

In [39]:
ncr_event["Supplier Name"].notna().mean()

0.999855637361051

In [40]:
#check 
ncr_oc.groupby("Notification Number")["Supplier Name"].nunique().value_counts()

Supplier Name
1    20778
0        3
Name: count, dtype: int64

In [41]:
ncr_event["Supplier Name"].value_counts().head(20)

Supplier Name
CRRC Changchun Railway Vehicle    4734
WABTEC Passenger Transit Corpo    3007
Vapor Stone Rail Systems          1673
Mitsubishi Electric Power Prod    1269
TOA Engineering Corporation       1034
TDG Transit Design Group INTL      663
Baoying ShenFei Electronic Equ     645
Bossard LLC                        400
Fastenal Co.                       378
Changchun Lu Tong                  365
BBA Project, Inc.                  318
Vulcan Metals Corporation          317
CRRC Times Electric USA, LLC       316
Faiveley Transportation            308
CRRC Qingdao Sifang Rolling St     300
WhiteStone Partners, IncDBA Go     296
CRRC MA Corporation                278
Kustom Seating Unlimited, Inc.     230
Nanjing Kangni Mechanical And      205
Qingdao SRI Technology Co., Lt     202
Name: count, dtype: int64

In [42]:
ncr_event["Supplier Name"].nunique()

119

In [43]:
# Generate NCR_month dataset
ncr_event["event_month"] = pd.to_datetime(
    ncr_event["Notification Created Date"]
).dt.to_period("M")

In [44]:
# Aggregated datasets by Supplier*Month
ncr_month = (
    ncr_event
    .groupby(["Supplier Name", "event_month"])
    .agg(
        ncr_count=("Notification Number", "count"),
        severe_ncr=("severity", lambda x: (x >= 3).sum()),
        total_loss=("Material Value", "sum")
    )
    .reset_index()
)

In [ ]:
#check
ncr_month.shape

(2933, 5)

In [46]:
ncr_month.head()

,Supplier Name,event_month,ncr_count,severe_ncr,total_loss
0,3m Company,2023-01,1,1,1016.40
1,ABB INC.,2023-10,1,0,138280.90
2,ABB INC.,2025-08,1,0,16.44
3,ABB INC.,2025-10,2,0,32.88
4,ACE Employment Unlimited Corp,2019-03,1,0,0.00


In [47]:
ncr_month = (
    ncr_event
    .groupby(["Supplier Name", "event_month"])
    .agg(
        ncr_count=("Notification Number", "count"),
        severe_ncr=("severity", lambda x: (x >= 3).sum()),
        total_loss=("Material Value", "sum")
    )
    .reset_index()
)

In [48]:
gr["event_month"] = pd.to_datetime(gr["Posting Date"]).dt.to_period("M")

In [50]:
gr_month = (
    gr
    .groupby(["Vendor", "event_month"])
    .agg(
        gr_qty=("Quantity", "sum"),
        gr_value=("Amt.in loc.cur.", "sum")
    )
    .reset_index()
)

In [52]:
ncr_event.head()

,Notification Number,Supplier Name,Notification Created Date,Notification Completion Date,Notification QTY,Material Value,severity,Cause Responsibility,event_month,cycle_days
0,200000001,WABTEC Passenger Transit Corpo,2019-02-19,2019-02-19,0,0.00,0,None,2019-02,0.0
1,200000012,Vulcan Metals Corporation,2019-03-01,2019-03-01,4,245781.36,0,None,2019-03,0.0
2,200000013,Vulcan Metals Corporation,2019-03-01,2019-03-01,4,245781.36,0,None,2019-03,0.0
3,200000017,ACE Employment Unlimited Corp,2019-03-05,2019-04-04,0,0.00,1,None,2019-03,30.0
4,200000032,TransTech of South Carolina In,2019-04-04,2019-04-16,1,66.01,4,None,2019-04,12.0


In [59]:
ncr_event["Notification Number"] = (
    ncr_event["Notification Number"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.strip()
)

ncr_all["Notification"] = (
    ncr_all["Notification"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.strip()
)

In [60]:
ncr_event = ncr_event.merge(
    ncr_all[["Notification", "Supplier"]],
    left_on="Notification Number",
    right_on="Notification",
    how="left"
)

In [62]:
ncr_event = ncr_event.rename(columns={"Supplier": "vendor_id"})

In [63]:
ncr_event[["Notification Number", "Supplier Name", "vendor_id"]].head(10)
ncr_event["vendor_id"].notna().mean()

0.4503633126413551

In [64]:
ncr_all["Notification"] = ncr_all["Notification"].astype(str).str.strip()
ncr_oc["Notification Number"] = ncr_oc["Notification Number"].astype(str).str.strip()

In [65]:
vendor_map = ncr_all[["Notification", "Supplier"]].drop_duplicates()

supplier_map = ncr_oc[["Notification Number", "Supplier Name"]].drop_duplicates()

In [66]:
ncr_map = supplier_map.merge(
    vendor_map,
    left_on="Notification Number",
    right_on="Notification",
    how="left"
)

In [67]:
ncr_map = ncr_map.rename(columns={"Supplier": "vendor_id"})
ncr_map = ncr_map.drop(columns=["Notification"])

In [68]:
ncr_map.head()

,Notification Number,Supplier Name,vendor_id
0,200029490,CRRC Qingdao Sifang Rolling St,2.000034e+09
1,200029482,Shenzhen HengZhiYuan Electric,NaN
2,200029481,Shijiazhuang King Transportati,NaN
3,200029479,"IFE North America, LLC",2.000033e+09
4,200029478,"CRRC Times Electric USA, LLC",NaN


In [69]:
ncr_map.groupby("Notification Number")[["Supplier Name","vendor_id"]].nunique()

,Supplier Name,vendor_id
Notification Number,,
200000001,1,1
200000012,1,0
200000013,1,0
200000017,1,0
200000032,1,1
...,...,...
200029478,1,0
200029479,1,1
200029481,1,0


In [70]:
ncr_map["vendor_id"].notna().mean()

0.4503633126413551

In [71]:
ncr_map.groupby("Supplier Name")["vendor_id"].nunique().sort_values(ascending=False).head(20)

Supplier Name
CRRC Qingdao Sifang Rolling St    2
TOA Engineering Corporation       1
Metro Customs Brokers Inc         1
Meeks, Sheppard, Leo & Pillsbu    1
Massachusetts Bay Associates,     1
Mass Signs LLC                    1
M+B Paul Inc                      1
Los Angeles County Metropolita    1
Lord Mechanical (Shanghai) Co.    1
Lord Corporation                  1
Kustom Seating Unlimited, Inc.    1
Kockum Sonics AB                  1
[DO NOT USE] USSC Group., Inc.    1
TransTech of South Carolina In    1
KSU, N.A., LLC                    1
KNORR BREMSE AUSTRALIA PTY LTD    1
KB Signaling, Inc.                1
Jiangsu Tie Mao Glass Co., Ltd    1
Transitair Systems, LLC           1
Intransit Technologies Corp.      1
Name: vendor_id, dtype: int64

In [72]:
ncr_event[["Notification Number","Supplier Name","vendor_id"]].head()

,Notification Number,Supplier Name,vendor_id
0,200000001,WABTEC Passenger Transit Corpo,2.000034e+09
1,200000012,Vulcan Metals Corporation,NaN
2,200000013,Vulcan Metals Corporation,NaN
3,200000017,ACE Employment Unlimited Corp,NaN
4,200000032,TransTech of South Carolina In,2.000034e+09


In [73]:
ncr_event["vendor_id"].notna().mean()

0.4503633126413551

In [74]:
ncr_event["vendor_id"] = (
    ncr_event["vendor_id"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.strip()
)

In [75]:
ncr_model = ncr_event[ncr_event["vendor_id"].notna()].copy()

In [76]:
ncr_model.shape

(20781, 12)

In [77]:
ncr_model["vendor_id"].nunique()

95

In [91]:
gr["Vendor"] = (
    gr["Vendor"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.strip()
)

In [92]:
ncr_month = (
    ncr_event
    .groupby(["vendor_id", "event_month"])
    .agg(
        ncr_count=("Notification Number", "count"),
        severe_ncr=("severity", lambda x: (x >= 3).sum()),
        total_loss=("Material Value", "sum")
    )
    .reset_index()
)

In [93]:
ncr_month.head()

,vendor_id,event_month,ncr_count,severe_ncr,total_loss
0,1000200000,2019-04,9,5,9364.93
1,1000200000,2019-05,16,8,11359.49
2,1000200000,2019-06,22,9,16550.85
3,1000200000,2019-07,9,9,571.59
4,1000200000,2019-08,5,2,4503.62


In [94]:
ncr_month.shape

(2184, 5)

In [95]:
gr_month = (
    gr
    .groupby(["Vendor", "event_month"])
    .agg(
        gr_qty=("Quantity", "sum"),
        gr_value=("Amt.in loc.cur.", "sum")
    )
    .reset_index()
    .rename(columns={"Vendor":"vendor_id"})
)

In [96]:
gr_month.head()

,vendor_id,event_month,gr_qty,gr_value
0,1000200000,2025-02,48450.00,-6784.47
1,1000200000,2025-03,61657.00,74623.44
2,1000200000,2025-04,19649.88,791778.37
3,1000200000,2025-05,24521.54,2196881.80
4,1000200000,2025-06,34621.88,3078624.57


In [97]:
gr_month.shape

(715, 4)

In [98]:
supplier_month = gr_month.merge(
    ncr_month,
    on=["vendor_id","event_month"],
    how="left"
)

In [99]:
supplier_month.head()

,vendor_id,event_month,gr_qty,gr_value,ncr_count,severe_ncr,total_loss
0,1000200000,2025-02,48450.00,-6784.47,7.0,6.0,715.70
1,1000200000,2025-03,61657.00,74623.44,18.0,3.0,477.87
2,1000200000,2025-04,19649.88,791778.37,10.0,5.0,2487.82
3,1000200000,2025-05,24521.54,2196881.80,10.0,6.0,1725.93
4,1000200000,2025-06,34621.88,3078624.57,5.0,4.0,195.36


In [100]:
supplier_month.shape

(715, 7)

In [101]:
supplier_month["ncr_rate"] = (
    supplier_month["ncr_count"] / supplier_month["gr_qty"]
)

supplier_month["severe_rate"] = (
    supplier_month["severe_ncr"] / supplier_month["gr_qty"]
)

In [103]:
supplier_month["loss_rate"] = (
    supplier_month["total_loss"] / supplier_month["gr_value"]
)

In [104]:
supplier_month.loc[
    supplier_month["gr_qty"] == 0,
    ["ncr_rate","severe_rate"]
] = 0

In [105]:
supplier_month.head()

,vendor_id,event_month,gr_qty,gr_value,ncr_count,severe_ncr,total_loss,ncr_rate,severe_rate,loss_rate
0,1000200000,2025-02,48450.00,-6784.47,7.0,6.0,715.70,0.000144,0.000124,-0.105491
1,1000200000,2025-03,61657.00,74623.44,18.0,3.0,477.87,0.000292,0.000049,0.006404
2,1000200000,2025-04,19649.88,791778.37,10.0,5.0,2487.82,0.000509,0.000254,0.003142
3,1000200000,2025-05,24521.54,2196881.80,10.0,6.0,1725.93,0.000408,0.000245,0.000786
4,1000200000,2025-06,34621.88,3078624.57,5.0,4.0,195.36,0.000144,0.000116,0.000063


In [109]:
supplier_month.to_csv("../1_data/03_processed/supplier_month_risk_base.csv",
    index=False
)

In [111]:
ncr_month.to_csv("../1_data/03_processed/ncr_month.csv", index=False)
gr_month.to_csv("../1_data/03_processed/gr_month.csv", index=False)